In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
torch.manual_seed(0)
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
import sys

torch.manual_seed(0)

In [2]:
# Setup vocabulary
vocab = [
    "[PAD]", "[MASK]", "[UNK]",

    "I", "you", "we", "they",

    # Polysemous words
    "bear",        # animal / tolerate
    "run",         # move / operate
    "bank",        # river / finance
    "charge",      # legal / electrical

    # Nouns
    "river", "road", "field", "court", "battery",
    "money", "load", "power", "side", "shore", "swam",

    # Function words
    "to", "the", "a", "other", "across", "with", "in", "on", "of", "get", "cannot"
]

vocab_size = len(vocab)
print(f'Vocabulary size is {vocab_size}')

Vocabulary size is 33


In [3]:
# Training data
training_data = [
    (["I", "swam", "across", "the", "river", "to", "the", "other", "[MASK]"], "shore"),
    (["they", "went", "to", "the", "bank", "to", "get", "[MASK]"], "money"),
    (["I", "saw", "a", "bear", "in", "the", "[MASK]"], "field"),
    (["I", "cannot", "bear", "the", "[MASK]"], "load"),
    (["they", "run", "across", "the", "[MASK]"], "field"),
    (["the", "battery", "can", "run", "with", "[MASK]"], "power"),
    (["the", "court", "will", "charge", "them", "with", "[MASK]"], "money"),
    (["the", "battery", "has", "a", "charge", "of", "[MASK]"], "power"),
]

In [4]:
sentence = training_data[0][0]
target = training_data[0][1]
idxtoword = {i:j for i,j in enumerate(vocab)}
wordtoidx = {j:i for i,j in enumerate(vocab)}
print(wordtoidx)
def encode(data):
    return torch.tensor([wordtoidx.get(i,wordtoidx["[UNK]"]) for i in data])

{'[PAD]': 0, '[MASK]': 1, '[UNK]': 2, 'I': 3, 'you': 4, 'we': 5, 'they': 6, 'bear': 7, 'run': 8, 'bank': 9, 'charge': 10, 'river': 11, 'road': 12, 'field': 13, 'court': 14, 'battery': 15, 'money': 16, 'load': 17, 'power': 18, 'side': 19, 'shore': 20, 'swam': 21, 'to': 22, 'the': 23, 'a': 24, 'other': 25, 'across': 26, 'with': 27, 'in': 28, 'on': 29, 'of': 30, 'get': 31, 'cannot': 32}


In [5]:
class Transformer(nn.Module):
    def __init__(self,vocab_size,embedingsize):
        super().__init__()

        self.wordEmbedings = nn.Embedding(vocab_size,embedingsize)
        self.q_w = nn.Linear(embedingsize,embedingsize,bias=False)
        self.k_w = nn.Linear(embedingsize,embedingsize,bias=False)
        self.v_w = nn.Linear(embedingsize,embedingsize,bias=False)

        self.W =  nn.Linear(embedingsize,vocab_size,bias=False)

    def forward(self,sentenceidx,maskidx):
        data = self.wordEmbedings(sentenceidx)
        # print(data)
        Q = self.q_w(data)
        K = self.k_w(data)
        V = self.v_w(data)
        S = F.softmax((Q@K.T)/math.sqrt(Q.size(-1)),dim=-1)

        ouput = S@V

        z = self.W(ouput)

        return z[maskidx]

def lossFun(logets,targetIdx):
    probs = F.softmax(logets,dim=-1)
    # print(probs)
    eps = 1e-09
    return -torch.log(probs[targetIdx]+eps)

In [6]:
vocab_size = len(vocab)
embedingsize = 8
epoch = 500
model = Transformer(vocab_size,embedingsize)
optimizer = torch.optim.Adam(model.parameters(),lr=0.05)
allLoss = []
for i in range(epoch):
    epochloss = 0.0
    random.shuffle(training_data)

    for sentence,target in training_data:
        sentenceidx = encode(sentence)
        # print(sentenceidx)
        maskidx = sentence.index("[MASK]")
        # print(maskidx)
        optimizer.zero_grad()
        targetIdx = wordtoidx[target]
        # print(targetIdx)

        logets = model(sentenceidx,maskidx)
        loss = lossFun(logets,targetIdx)
        loss.backward()
        optimizer.step()
        allLoss.append(loss.item())
        epochloss+=loss.item()
    if i%10==0:
        print(f"epoch {i} : {epochloss:4f}")


epoch 0 : 28.284336
epoch 10 : 0.068950
epoch 20 : 0.006944
epoch 30 : 0.003874
epoch 40 : 0.002530
epoch 50 : 0.001796
epoch 60 : 0.001349
epoch 70 : 0.001053
epoch 80 : 0.000842
epoch 90 : 0.000690
epoch 100 : 0.000576
epoch 110 : 0.000488
epoch 120 : 0.000419
epoch 130 : 0.000363
epoch 140 : 0.000317
epoch 150 : 0.000281
epoch 160 : 0.000249
epoch 170 : 0.000221
epoch 180 : 0.000199
epoch 190 : 0.000179
epoch 200 : 0.000162
epoch 210 : 0.000147
epoch 220 : 0.000134
epoch 230 : 0.000123
epoch 240 : 0.000112
epoch 250 : 0.000103
epoch 260 : 0.000095
epoch 270 : 0.000088
epoch 280 : 0.000081
epoch 290 : 0.000075
epoch 300 : 0.000070
epoch 310 : 0.000065
epoch 320 : 0.000061
epoch 330 : 0.000056
epoch 340 : 0.000053
epoch 350 : 0.000049
epoch 360 : 0.000046
epoch 370 : 0.000043
epoch 380 : 0.000041
epoch 390 : 0.000038
epoch 400 : 0.000036
epoch 410 : 0.000033
epoch 420 : 0.000032
epoch 430 : 0.000030
epoch 440 : 0.000028
epoch 450 : 0.000027
epoch 460 : 0.000025
epoch 470 : 0.000024
ep

In [69]:
no = np.random.randint(0,8)
print(np.random.randint(0,8))
sentence = training_data[no][0]
target = training_data[no][1]

print(sentence)
print(target)


sentenceidx = encode(sentence)
        # print(sentenceidx)
print(sentenceidx)
maskidx = sentence.index("[MASK]")
print(maskidx)

z = model(sentenceidx,maskidx)
probs = F.softmax(z,dim=-1)

print(f"pridiction : {idxtoword[torch.argmax(probs).item()]}")


1
['the', 'court', 'will', 'charge', 'them', 'with', '[MASK]']
money
tensor([23, 14,  2, 10,  2, 27,  1])
6
pridiction : money
